In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

In [2]:
# Load the model
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Print the model architecture
print(model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [3]:
# Print the model configuration
print(model.config)

BertConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "sentence-transformers/all-MiniLM-L6-v2",
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.46.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [4]:
# Input sentences
sentence1 = "quả cam ngon ."
sentence2 = "quả táo dở ."

In [5]:
# Tokenize each sentence separately
def get_encoding(sentence):
    encoding = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True, max_length=512)
    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']
    token_type_ids = encoding.get('token_type_ids', torch.zeros_like(input_ids))  # MiniLM may not use token_type_ids
    return input_ids, attention_mask, token_type_ids

In [6]:
# Compute embeddings
def get_embedding(input_ids, attention_mask):
    output = model(input_ids=input_ids, attention_mask=attention_mask)
    embeddings = output['last_hidden_state']
    attention_mask = attention_mask.unsqueeze(-1)  # Expand mask for broadcasting
    
    # Mean pooling (ignoring padding tokens)
    sentence_embedding = torch.sum(embeddings * attention_mask, dim=1) / attention_mask.sum(dim=1)
    
    # Normalize embeddings to unit vectors
    return F.normalize(sentence_embedding, p=2, dim=1)

In [7]:
# Get encodings for both sentences
input_ids1, attention_mask1, token_type_ids1 = get_encoding(sentence1)
input_ids2, attention_mask2, token_type_ids2 = get_encoding(sentence2)

# Convert tokens to human-readable format
tokens1 = tokenizer.convert_ids_to_tokens(input_ids1.squeeze().tolist())
tokens2 = tokenizer.convert_ids_to_tokens(input_ids2.squeeze().tolist())

In [8]:
# Display tokenized results
print(f"\nInput Sentence 1: {sentence1}")
print(f"Tokens: {tokens1}")
print(f"Token IDs: {input_ids1.squeeze().tolist()}")
print(f"Attention Mask: {attention_mask1.squeeze().tolist()}")
print(f"Token Type IDs: {token_type_ids1.squeeze().tolist()}")

print(f"\nInput Sentence 2: {sentence2}")
print(f"Tokens: {tokens2}")
print(f"Token IDs: {input_ids2.squeeze().tolist()}")
print(f"Attention Mask: {attention_mask2.squeeze().tolist()}")
print(f"Token Type IDs: {token_type_ids2.squeeze().tolist()}")


Input Sentence 1: quả cam ngon .
Tokens: ['[CLS]', 'qu', '##a', 'cam', 'ngo', '##n', '.', '[SEP]']
Token IDs: [101, 24209, 2050, 11503, 17895, 2078, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1]
Token Type IDs: [0, 0, 0, 0, 0, 0, 0, 0]

Input Sentence 2: quả táo dở .
Tokens: ['[CLS]', 'qu', '##a', 'tao', 'do', '.', '[SEP]']
Token IDs: [101, 24209, 2050, 20216, 2079, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1]
Token Type IDs: [0, 0, 0, 0, 0, 0, 0]


In [9]:
# Check max position embeddings
max_position_embeddings = model.config.max_position_embeddings  # Typically 512 for MiniLM
print(f"\nMax Position Embeddings: {max_position_embeddings}")


Max Position Embeddings: 512


In [10]:
# Ensure input length doesn't exceed max position embeddings
input_length1 = input_ids1.shape[1]
input_length2 = input_ids2.shape[1]
print(f"Input Sequence Length (Sentence 1): {input_length1}")
print(f"Input Sequence Length (Sentence 2): {input_length2}")

Input Sequence Length (Sentence 1): 8
Input Sequence Length (Sentence 2): 7


In [12]:
if input_length1 > max_position_embeddings:
    print(f"Warning: Sentence 1 exceeds max position embeddings. Truncating sequence.")
    input_ids1 = input_ids1[:, :max_position_embeddings]
    attention_mask1 = attention_mask1[:, :max_position_embeddings]

if input_length2 > max_position_embeddings:
    print(f"Warning: Sentence 2 exceeds max position embeddings. Truncating sequence.")
    input_ids2 = input_ids2[:, :max_position_embeddings]
    attention_mask2 = attention_mask2[:, :max_position_embeddings]

# Compute embeddings
embedding1 = get_embedding(input_ids1, attention_mask1)
embedding2 = get_embedding(input_ids2, attention_mask2)

# Compute cosine similarity
cosine_sim = F.cosine_similarity(embedding1, embedding2, dim=1)

# Print similarity score
print(f"\nCosine Similarity: {cosine_sim.item():.4f}")


Cosine Similarity: 0.5184


In [13]:
# Inspect individual embeddings
word_embeddings = model.embeddings.word_embeddings
position_embeddings = model.embeddings.position_embeddings
token_type_embeddings = model.embeddings.token_type_embeddings if hasattr(model.embeddings, 'token_type_embeddings') else None

In [14]:
print(f"\n--- Word Embeddings ---")
print(f"Word Embeddings Shape: {word_embeddings.weight.shape}")
print(f"Word Embeddings Example (for first sentence): {word_embeddings(input_ids1).squeeze(0).detach().cpu().numpy()[:5]}")  # Show first 5 tokens for brevity


--- Word Embeddings ---
Word Embeddings Shape: torch.Size([30522, 384])
Word Embeddings Example (for first sentence): [[-0.0176194  -0.00760055  0.04710554 ... -0.05453258  0.0075766
  -0.06167737]
 [ 0.0052347   0.00371009 -0.06323255 ... -0.00576439 -0.05213243
   0.00914592]
 [-0.05324243 -0.02380681 -0.05003943 ...  0.00206165 -0.04402293
   0.11747763]
 [ 0.06209818 -0.00330933  0.02244107 ...  0.00408579 -0.15784408
  -0.05605971]
 [ 0.0839615  -0.05400614 -0.05587685 ...  0.00022017  0.01230263
  -0.04707626]]


In [15]:
print(f"\n--- Position Embeddings ---")
print(f"Position Embeddings Shape: {position_embeddings.weight.shape}")
position_indices = torch.arange(input_length1).unsqueeze(0)
position_embedding_example = position_embeddings(position_indices)
print(f"Position Embeddings Example: {position_embedding_example.squeeze(0).detach().cpu().numpy()[:5]}")  # Show first 5 tokens


--- Position Embeddings ---
Position Embeddings Shape: torch.Size([512, 384])
Position Embeddings Example: [[-0.08555417 -0.03291567 -0.01702809 ...  0.08736189  0.09658185
   0.02673073]
 [-0.03272121  0.00227195  0.02110414 ...  0.02815887  0.04547747
  -0.00984178]
 [-0.01342716 -0.00950335  0.03170437 ...  0.00633396  0.04613364
  -0.01627073]
 [-0.02343853  0.00066019  0.02879668 ... -0.01339278  0.04062235
  -0.02393162]
 [-0.03131729  0.01603579  0.03266083 ... -0.01082838  0.03790909
  -0.02438987]]


In [16]:
if token_type_embeddings is not None:
    print(f"\n--- Token Type Embeddings ---")
    print(f"Token Type Embeddings Shape: {token_type_embeddings.weight.shape}")
    print(f"Token Type Embeddings Example (for sentence 1): {token_type_embeddings(token_type_ids1).squeeze(0).detach().cpu().numpy()[:5]}")


--- Token Type Embeddings ---
Token Type Embeddings Shape: torch.Size([2, 384])
Token Type Embeddings Example (for sentence 1): [[ 0.01461648  0.00376141 -0.01204102 ... -0.00675753 -0.01298677
   0.0197649 ]
 [ 0.01461648  0.00376141 -0.01204102 ... -0.00675753 -0.01298677
   0.0197649 ]
 [ 0.01461648  0.00376141 -0.01204102 ... -0.00675753 -0.01298677
   0.0197649 ]
 [ 0.01461648  0.00376141 -0.01204102 ... -0.00675753 -0.01298677
   0.0197649 ]
 [ 0.01461648  0.00376141 -0.01204102 ... -0.00675753 -0.01298677
   0.0197649 ]]


In [17]:
# Display encoder details
encoder_output1 = model(input_ids=input_ids1, attention_mask=attention_mask1)['last_hidden_state']
encoder_output2 = model(input_ids=input_ids2, attention_mask=attention_mask2)['last_hidden_state']

print(f"\n--- Encoder Output ---")
print(f"Encoder Output Shape (Sentence 1): {encoder_output1.shape}")
print(f"Encoder Output Shape (Sentence 2): {encoder_output2.shape}")


--- Encoder Output ---
Encoder Output Shape (Sentence 1): torch.Size([1, 8, 384])
Encoder Output Shape (Sentence 2): torch.Size([1, 7, 384])


In [18]:
# Inspect individual encoder layers
print(f"\n--- Encoder Layer Details ---")
for i, layer in enumerate(model.encoder.layer):
    print(f"\nLayer {i+1} Details:")
    
    # Attention details
    self_attention = layer.attention.self
    print(f"Self-Attention (Layer {i+1})")
    print(f"Query Projection Weights: {self_attention.query.weight.shape}")
    print(f"Key Projection Weights: {self_attention.key.weight.shape}")
    print(f"Value Projection Weights: {self_attention.value.weight.shape}")
    print(f"Output Projection Weights: {layer.attention.output.dense.weight.shape}")

    # Feedforward details
    intermediate = layer.intermediate
    output_layer = layer.output
    print(f"Feedforward Layer (Layer {i+1})")
    print(f"Intermediate Layer Weights: {intermediate.dense.weight.shape}")
    print(f"Output Layer Weights: {output_layer.dense.weight.shape}")


--- Encoder Layer Details ---

Layer 1 Details:
Self-Attention (Layer 1)
Query Projection Weights: torch.Size([384, 384])
Key Projection Weights: torch.Size([384, 384])
Value Projection Weights: torch.Size([384, 384])
Output Projection Weights: torch.Size([384, 384])
Feedforward Layer (Layer 1)
Intermediate Layer Weights: torch.Size([1536, 384])
Output Layer Weights: torch.Size([384, 1536])

Layer 2 Details:
Self-Attention (Layer 2)
Query Projection Weights: torch.Size([384, 384])
Key Projection Weights: torch.Size([384, 384])
Value Projection Weights: torch.Size([384, 384])
Output Projection Weights: torch.Size([384, 384])
Feedforward Layer (Layer 2)
Intermediate Layer Weights: torch.Size([1536, 384])
Output Layer Weights: torch.Size([384, 1536])

Layer 3 Details:
Self-Attention (Layer 3)
Query Projection Weights: torch.Size([384, 384])
Key Projection Weights: torch.Size([384, 384])
Value Projection Weights: torch.Size([384, 384])
Output Projection Weights: torch.Size([384, 384])
Fee

In [19]:
# Pooler Layer (ensure it's available and print the output)
if hasattr(model, 'pooler'):
    pooler_output1 = model(input_ids=input_ids1, attention_mask=attention_mask1)['pooler_output']
    pooler_output2 = model(input_ids=input_ids2, attention_mask=attention_mask2)['pooler_output']

    print(f"\n--- Pooler Layer ---")
    print(f"Pooler Output Shape (Sentence 1): {pooler_output1.shape}")
    print(f"Pooler Output Shape (Sentence 2): {pooler_output2.shape}")
    print(f"Pooler Output Example (Sentence 1): {pooler_output1}")
    print(f"Pooler Output Example (Sentence 2): {pooler_output2}")
    print(f"Pooler Dense Layer: {model.pooler.dense}")
    print(f"Pooler Activation: {model.pooler.activation}")
else:
    print("\nNo Pooler Layer found in the model.")


--- Pooler Layer ---
Pooler Output Shape (Sentence 1): torch.Size([1, 384])
Pooler Output Shape (Sentence 2): torch.Size([1, 384])
Pooler Output Example (Sentence 1): tensor([[-4.3994e-02,  4.6173e-02,  8.1268e-02,  9.0341e-02,  4.4443e-02,
         -5.8203e-02,  1.8319e-02,  9.9856e-02, -2.5031e-03,  1.8227e-02,
         -6.4416e-03, -2.7775e-03, -4.7234e-02, -2.0300e-02, -3.4868e-02,
          6.9777e-02,  7.3711e-02,  4.8582e-02,  7.9299e-03,  1.7280e-03,
          5.2632e-02,  1.1334e-02, -4.8164e-03, -3.4766e-02,  1.0420e-02,
          6.9434e-02,  9.8169e-03,  2.6915e-02,  6.1287e-02,  1.0119e-01,
          4.9672e-02,  2.4097e-02, -1.2251e-01,  8.2956e-02,  3.7269e-02,
         -6.0411e-02, -1.6368e-03,  5.1087e-02, -4.5087e-02, -6.2142e-02,
          5.8030e-02, -1.1793e-02,  8.0903e-03,  1.0928e-01, -9.0935e-02,
          4.5734e-02, -1.1006e-01, -4.4590e-02,  3.8400e-02, -2.1706e-02,
         -1.0779e-01,  1.1638e-01,  6.5873e-02, -9.7565e-03, -5.0636e-02,
         -8.2013e-